# NLP Assignment 1: Google Colab Build

**Assignment:** NLP Assignment 1 (10%), Addis Ababa University, School of Information Technology and Engineering.
**Instructor:** Fantahun B. (PhD)
**Questions Covered:**
- Step 0: Raw corpus verification and assessment, cleaned text generation, verification, vocabulary and token array creation.
- Q1.1 to Q1.8: N-gram language models, MLE probabilities, smoothing, conditional probabilities, stopword analysis, sentence scoring, and model generation.
- Q2: Intrinsic evaluation (perplexity across orders 1 to 4 on held-out test splits, primary seed 42 and replication seed 7).
- Q3: Extrinsic evaluation (real vs corrupted Amharic sentence detection, threshold tuning, UNK confound analysis, ROC-AUC).
- Q4: Predictive text application evaluation (Top-1, Top-5, MRR@5 across 1-word and 2-word contexts).
- Presentation Artifacts: Canonical artifact bundle generation for local PC exploration.

**Execution Rule (AGENTS.md B13):**
Heavy pipeline stages run exclusively in this heavy environment. The local PC runs only unit tests, checksum verification (`scripts/check_download.py`), and the light interactive UI/CLI.

In [15]:
import os, sys, platform, importlib, json, shutil

def read_first(path):
    try:
        return open(path).read().strip()
    except Exception:
        return None

cg = read_first("/sys/fs/cgroup/memory.max") or read_first("/sys/fs/cgroup/memory/memory.limit_in_bytes")
try:
    phys = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
except Exception:
    phys = 0

print("Google Colab detected =", "google.colab" in sys.modules or os.path.exists("/content"))
print("physical RAM (GiB)    =", round(phys / 2**30, 2) if phys else "N/A")
print("cgroup limit         =", cg)
print("CPUs                 =", os.cpu_count())
print("disk free (GiB)      =", round(shutil.disk_usage("/content").free / 2**30, 1) if os.path.isdir("/content") else "N/A")
print("python               =", platform.python_version())

Google Colab detected = True
physical RAM (GiB)    = 12.67
cgroup limit         = max
CPUs                 = 2
disk free (GiB)      = 76.8
python               = 3.13.16


For a larger budget, choose *Runtime → Change runtime type → High-RAM*, if your account offers it.

In [16]:
!pip install -q gdown
import sklearn, numpy
print("scikit-learn version:", sklearn.__version__)
print("numpy version:       ", numpy.__version__)

scikit-learn version: 1.6.1
numpy version:        2.1.3


Dependencies verified.

In [17]:
import os

# Start from a directory that always exists, avoiding getcwd errors
os.chdir("/content")

REPO_URL = "https://github.com/KenaKaye/Assignment-1.git"
BRANCH = "full-corpus"

!rm -rf /content/NLP_Assignment1
!git clone -b {BRANCH} {REPO_URL} /content/NLP_Assignment1
%cd /content/NLP_Assignment1
!git log -1 --oneline

Cloning into '/content/NLP_Assignment1'...
remote: Enumerating objects: 347, done.
remote: Counting objects: 100% (347/347), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 347 (delta 155), reused 279 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (347/347), 13.92 MiB | 26.01 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/NLP_Assignment1
5204896 (HEAD -> full-corpus, origin/full-corpus) feat: filter sentence boundary markers in Q1.3, standardize CLI and Web UI on shared pipeline engine, and include training counts in Colab export bundle


Record the commit printed here in `README.md`.

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Run the cell above to connect your Google Drive containing `GPAC.txt.gz`.

In [19]:
import gzip, shutil, hashlib, os, glob

# Start from a directory that always exists, then enter the repository
os.chdir("/content")
os.chdir("/content/NLP_Assignment1")

SRC = "/content/drive/MyDrive/GPAC.txt.gz"
if not os.path.exists(SRC):
    candidates = glob.glob("/content/drive/*/**/GPAC.txt.gz", recursive=True) + glob.glob("/content/GPAC.txt.gz")
    if candidates:
        SRC = candidates[0]
        print(f"Found GPAC.txt.gz at {SRC}")
    else:
        raise SystemExit("GPAC.txt.gz not found at /content/drive/MyDrive/GPAC.txt.gz. Please check Drive path.")

DST = "/content/NLP_Assignment1/data/GPAC.txt"
os.makedirs(os.path.dirname(DST), exist_ok=True)

print(f"Decompressing {SRC} -> {DST}...")
with gzip.open(SRC, "rb") as src, open(DST, "wb") as dst:
    shutil.copyfileobj(src, dst)

h = hashlib.sha256()
with open(DST, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)

actual_hash = h.hexdigest()
EXPECTED = "977706859188f7598ca97a29d7203d86926c11ede591fba31915c74ae9eabcfa"
if actual_hash != EXPECTED:
    print("----------------------------------------------------------------------")
    print(f"ACTUAL HASH: {actual_hash}")
    print(f"EXPECTED:    {EXPECTED}")
    print("The attached file does not match the authoritative corpus hash.")
    print("----------------------------------------------------------------------")

assert actual_hash == EXPECTED, f"Raw corpus hash mismatch: {actual_hash}"
print("GPAC.txt SHA-256 matches:", actual_hash)

Decompressing /content/drive/MyDrive/GPAC.txt.gz -> /content/NLP_Assignment1/data/GPAC.txt...
GPAC.txt SHA-256 matches: 977706859188f7598ca97a29d7203d86926c11ede591fba31915c74ae9eabcfa


The raw corpus hash strictly matches the authoritative value recorded in the implementation plan. Step 0 raw data verification passes. (If the assert fails, the file is wrong).

In [20]:
import os, shutil
from pathlib import Path

os.chdir("/content/NLP_Assignment1")

RESET_STAGES = []

cleaned_file = Path("data/GPAC_cleaned.txt")
if cleaned_file.exists():
    with open(cleaned_file, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
        if first_line and not any(first_line.endswith(p) for p in ["።", "?", "!"]):
            print("Detected stale GPAC_cleaned.txt lacking sentence-final punctuation! Resetting...")
            RESET_STAGES = ["02_build_cleaned"]

if RESET_STAGES:
    print(f"Resetting pipeline stages starting from {RESET_STAGES}...")
    manifest_file = Path("data/manifest.json")
    if manifest_file.exists():
        manifest_file.unlink()
    if cleaned_file.exists():
        cleaned_file.unlink()
    shutil.rmtree("data/counts", ignore_errors=True)
    shutil.rmtree("results", ignore_errors=True)
    for f in Path("data").glob("*.npy"):
        f.unlink()
    for f in Path("data").glob("*.tsv"):
        if f.name != "sentence_assessment.tsv":
            f.unlink()
    print("Reset complete.")
else:
    print("Manifest checkpoints active.")

Manifest checkpoints active.


Old outputs verified.

In [21]:
import subprocess, time, sys, os, json

os.chdir("/content/NLP_Assignment1")

STAGES = [
    ["scripts/01_assess_raw.py"],
    ["scripts/02_build_cleaned.py"],
    ["scripts/03_verify_cleaned.py"],
    ["scripts/04_build_vocab_tokens.py"],
    ["scripts/06_split.py"],
    ["scripts/07_count_full.py"],
    ["scripts/08_count_train.py", "--seed", "42"],
    ["scripts/08_count_train.py", "--seed", "7"],
    ["scripts/10_q1_results.py"],
    ["scripts/11_q2_perplexity.py"],
    ["scripts/12_q3_extrinsic.py"],
    ["scripts/13_q4_eval.py"],
]

times = {}
os.makedirs("results", exist_ok=True)

for args in STAGES:
    name = " ".join(args)
    print(f"\n==================== Running: {name} ====================", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        [sys.executable, "-u", *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    dur = round(time.time() - t0, 1)
    times[name] = dur
    print(f"== {name}: exit code {rc}, elapsed: {dur} s ==", flush=True)
    if rc != 0:
        raise RuntimeError(f"Pipeline Stage Failed: {name} (exit code {rc})")

with open("results/colab_stage_times.json", "w", encoding="utf-8") as f:
    json.dump(times, f, indent=2)

print("\n[PIPELINE COMPLETE] All 12 stages finished successfully!")


==================== Running: scripts/01_assess_raw.py ====================
[REQUIRE_HEAVY] Verified Colab runtime for: 01_assess_raw
--- Starting Stage: 01_assess_raw ---
  Processed 509,461 segments... (valid=494,721)
  Processed 1,012,376 segments... (valid=987,598)
  Processed 1,503,401 segments... (valid=1,468,960)
  Processed 2,012,982 segments... (valid=1,967,557)
  Processed 2,502,015 segments... (valid=2,439,458)
  Processed 3,006,781 segments... (valid=2,932,788)
  Processed 3,510,701 segments... (valid=3,430,091)
  Processed 4,012,754 segments... (valid=3,924,814)
  Processed 4,519,956 segments... (valid=4,356,724)
  Processed 5,023,744 segments... (valid=4,796,643)

--- Assessment Completed ---
Elapsed Time: 93.21 s
Peak Memory: 446.07 MB
Total Segments: 5,255,013
Status Breakdown: {'valid': 4993573, 'too_short': 62740, 'empty': 195971, 'non_ethiopic': 2729}
Total Valid Sentences: 4,993,573
Total Valid Word Tokens: 76,167,531
Delimiter Counts: {'።': 3113238, '?': 488345, '

Pipeline execution completed.

In [22]:
import os, json
import numpy as np

os.chdir("/content/NLP_Assignment1")

print("--- Running Local Unit Test Suite ---")
!python -m pytest tests/

print("\n--- Pipeline Gate Checks ---")
v_rec = json.load(open("data/cleaned_verification_record.json"))
letter = v_rec.get("ethiopic_letter_share_pct", v_rec.get("ethiopic_letters_only_share"))
punct = v_rec.get("ethiopic_with_punctuation_share_pct", v_rec.get("ethiopic_letters_punct_share"))
print("Ethiopic letter share:       ", f"{letter:.2f}% (Benchmark: > 80%)")
print("Ethiopic with punct share:   ", f"{punct:.2f}%")
assert letter >= 80.0

tokens_len = len(np.load("data/tokens.npy", mmap_mode="r"))
for n in range(1, 5):
    c = np.load(f"data/counts/full/order{n}_counts.npy")
    assert c.sum() == tokens_len
    p_sum = (c / tokens_len).sum()
    diff = abs(p_sum - 1.0)
    print(f"  Order {n}: counts sum = {c.sum():,} | prob sum = {p_sum:.10f} (diff = {diff:.2e})")
    assert diff <= 1e-9

gen_data = json.load(open("results/q1_generation.json"))
print("\nQ1.8 Generation Verbatim Copies:")
for order_k, sents in gen_data.items():
    verb_cnt = sum(1 for s in sents if s["is_full_verbatim_copy"])
    print(f"  {order_k}: {verb_cnt} / {len(sents)} sentences are verbatim corpus copies")

q2_data = json.load(open("results/q2_perplexity.json"))
print("\nQ2 Held-out Evaluation:")
print(f"  Seed 42 UNK fraction: {q2_data['seed_42']['unk_fraction'] * 100:.2f}%")
print(f"  Seed 7 UNK fraction:  {q2_data['seed_7']['unk_fraction'] * 100:.2f}%")
print(f"  Seed 42 Bigram PP Reduction: {q2_data['seed_42']['orders']['n=2']['reduction_vs_unigram_pct']:.2f}%")

print("\n[ALL GATE CHECKS PASSED]")

--- Running Local Unit Test Suite ---
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/NLP_Assignment1
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 24 items                                                             

tests/test_api.py ......                                                 [ 25%]
tests/test_specification.py .............                                [ 79%]
tests/test_step0_vectorization.py .....                                  [100%]

======================== 24 passed in 154.26s (0:02:34) ========================

--- Pipeline Gate Checks ---
Ethiopic letter share:        97.86% (Benchmark: > 80%)
Ethiopic with punct share:    100.00%
  Order 1: counts sum = 81,161,104 | prob sum = 1.0000000000 (diff = 2.55e-15)
  Order 2: counts sum = 81,161,104 | prob sum = 1.0000000000 (diff = 4.11e-15)
  Order 3: counts sum 

All gate checks pass.

In [23]:
import os, json
import pandas as pd

os.chdir("/content/NLP_Assignment1")

print("=== Q1 Top 5 Bigrams (Q1.3) ===")
print(pd.read_csv("results/q1_top10_n2.csv").head(5).to_string(index=False))

print("\n=== Q1 Conditional Probabilities (Q1.4) ===")
print(pd.read_csv("results/q1_bigram_conditional.csv").to_string(index=False))

print("\n=== Q1 Top 5 Bigrams After Stopword Removal (Q1.5) ===")
print(pd.read_csv("results/q1_stopword_top10_n2.csv").head(5).to_string(index=False))

print("\n=== Q2 Perplexity Summary ===")
q2 = json.load(open("results/q2_perplexity.json"))
for seed_name in ["seed_42", "seed_7"]:
    print(f"  -- {seed_name} --")
    for ok, ov in q2[seed_name]["orders"].items():
        print(f"     {ok}: PP = {ov['perplexity']}, LogP = {ov['logp_sum']}, Reduction = {ov['reduction_vs_unigram_pct']}%")

print("\n=== Q3 Extrinsic Metrics ===")
q3 = json.load(open("results/q3_metrics.json"))
for ok, ov in q3["models"].items():
    print(f"  {ok}: Acc={ov['accuracy']}, F1={ov['f1_score']}, AUC={ov['roc_auc']}, Usable={ov['usable']}")

print("\n=== Q4 Predictive Text Evaluation ===")
q4 = json.load(open("results/q4_eval.json"))
for mk, mv in q4["metrics"].items():
    print(f"  {mk}: Top-1={mv['top1_accuracy']}, Top-5={mv['top5_hit_rate']}, MRR@5={mv['mrr_at_5']}, Meets Target={mv['meets_benchmarks']}")
print("\nExample Prompt Suggestion ('ኢትዮጵያ ውስጥ የቡና'):")
for s in q4["example_prompt_output"]["suggestions"]:
    print(f"  {s['word']:15} | Prob: {s['prob']:.4f} | Train Count: {s['train_count']} | Verified: {s['verified_in_corpus']}")

=== Q1 Top 5 Bigrams (Q1.3) ===
 rank  ngram  count  probability
    1    ዓ ም 108455     0.001336
    2   ቀን ዓ  46525     0.000573
    3 ነገር ግን  44744     0.000551
    4    ዶ ር  35624     0.000439
    5 ማለት ነው  32386     0.000399

=== Q1 Conditional Probabilities (Q1.4) ===
   w1   w2  count_w1  count_w1_w2  P(w2|w1)
ኢትዮጵያ  ውስጥ    102573         8181  0.079758
  አዲስ  አበባ     88897        31388  0.353083
መንግስት   እና     78628          474  0.006028
  ሰማይ ድንጋይ      6159            0  0.000000

=== Q1 Top 5 Bigrams After Stopword Removal (Q1.5) ===
 rank   ngram  count
    1     ዓ ም 108455
    2    ቀን ዓ  46525
    3     ዶ ር  35624
    4 አዲስ አበባ  31388
    5     በ ዓ  28486

=== Q2 Perplexity Summary ===
  -- seed_42 --
     n=1: PP = 16317.94, LogP = -78867248.2, Reduction = 0.0%
     n=2: PP = 1627.11, LogP = -60122420.92, Reduction = 90.03%
     n=3: PP = 921.78, LogP = -55502189.84, Reduction = 94.35%
     n=4: PP = 648.51, LogP = -52643189.81, Reduction = 96.03%
  -- seed_7 --
     n=1:

Results summary displayed.

In [24]:
import os, shutil, json, sys

os.chdir("/content/NLP_Assignment1")

sys.path.insert(0, ".")
from scripts.artifacts import write_artifact_list
write_artifact_list()

EXPORT = "/content/export"
os.makedirs(EXPORT, exist_ok=True)

ARTIFACTS = json.load(open("data/artifact_list.json"))
total_bytes = 0
copied_count = 0

for rel in ARTIFACTS:
    src = os.path.join("/content/NLP_Assignment1", rel)
    dst = os.path.join(EXPORT, rel)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        total_bytes += os.path.getsize(dst)
        copied_count += 1

print(f"Exported {copied_count} canonical artifacts: {total_bytes:,} bytes ({round(total_bytes / 2**20, 2)} MiB)")
shutil.make_archive("/content/nlp_assignment1_artifacts", "zip", EXPORT)
print("Created /content/nlp_assignment1_artifacts.zip")

Exported 32 canonical artifacts: 621,217,975 bytes (592.44 MiB)
Created /content/nlp_assignment1_artifacts.zip


Artifacts bundled into zip.

In [25]:
import os, shutil

os.chdir("/content/NLP_Assignment1")

for fname in ["checksums.json", "manifest.json", "artifact_list.json"]:
    p = os.path.join("data", fname)
    if os.path.exists(p):
        shutil.copy2(p, os.path.join(EXPORT, "data", fname))
        print(f"Included {fname} in export.")

log_p = os.path.join("results", "run_log.jsonl")
if os.path.exists(log_p):
    shutil.copy2(log_p, os.path.join(EXPORT, "results", "run_log.jsonl"))
    print("Included run_log.jsonl in export.")

print("Export checksum verification bundle ready.")

Included checksums.json in export.
Included manifest.json in export.
Included artifact_list.json in export.
Included run_log.jsonl in export.
Export checksum verification bundle ready.


Checksum bundle exported.

### Next Steps: PC Download and Local Verification

1. Download `nlp_assignment1_artifacts.zip` from Colab files pane.
2. Extract the files into your local repository root on your PC:
   - `data/` artifacts overwrite `data/`
   - `results/` artifacts overwrite `results/`
3. On your local PC, run:
   ```bash
   python scripts/check_download.py
   ```
4. Once `check_download.py` verifies all SHA-256 hashes, launch the local interactive UI and CLI:
   ```bash
   python app/cli.py
   ```
   Or run the local web interface (`web/` via `npm run dev`).

In [26]:
import shutil

shutil.copy2("/content/nlp_assignment1_artifacts.zip", "/content/drive/MyDrive/nlp_assignment1_artifacts.zip")
print("Successfully copied to your Google Drive: nlp_assignment1_artifacts.zip")


Successfully copied to your Google Drive: nlp_assignment1_artifacts.zip
